# Notebook 21 — PEFT: LoRA applicato a GPT-2

**Modulo:** 21 — Allineamento e generazione multimodale
**Tempo stimato:** 1,5-2 ore (eseguibile su CPU, non serve GPU)

## Cosa costruirai

Al termine di questo notebook avrai:
- Contato "a mano" (via codice) i parametri totali e addestrabili di un modello prima e dopo aver applicato LoRA
- Configurato `LoraConfig` con la libreria `peft` di Hugging Face su GPT-2 (124M parametri)
- Verificato, con un calcolo numerico riproducibile, il risparmio di parametri che LoRA introduce sulla matrice di attenzione reale del modello
- Addestrato un adattatore LoRA su un mini-dataset scritto in italiano, per uno stile editoriale (titolo + parole chiave SEO + riassunto in JSON)
- Confrontato le generazioni del modello prima/dopo il fine-tuning
- Salvato e ricaricato l'adattatore, verificando che occupi pochi megabyte contro le centinaia di megabyte del modello base

## Prerequisiti

```bash
pip install transformers peft accelerate torch datasets python-dotenv
```

Non serve una GPU: GPT-2 (124M parametri) e un dataset di una decina di esempi si allenano
in pochi minuti anche su CPU. Non serve nemmeno un file `.env` per questo notebook (GPT-2 è un
modello pubblico), ma manteniamo comunque il pattern di verifica delle chiavi usato nel resto
del corso, per il caso in cui tu voglia in futuro caricare un modello con accesso ristretto
dall'Hugging Face Hub.

> **Nota:** questo notebook usa GPT-2 invece di un modello della famiglia LLaMA (come nel caso
> pratico della dispensa, Sezione 5) proprio perché deve girare senza hardware dedicato. La
> logica di LoRA è identica; cambiano solo le dimensioni del modello e, di conseguenza, le
> percentuali esatte che osserverai (vedi la Sezione "Esperimento guidato" più sotto).

---
## Cella 0 — Setup

Carichiamo un eventuale token Hugging Face da `.env` (opzionale per questo notebook: GPT-2 è
pubblico) e verifichiamo quale device useremo.

In [ ]:
import os
from dotenv import load_dotenv
import torch

load_dotenv()

HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")  # opzionale per questo notebook

def verifica_chiave(nome: str, valore: str | None) -> None:
    if valore:
        print(f"OK {nome}: {valore[:8]}...")
    else:
        print(f"NON IMPOSTATA {nome} (va bene per questo notebook, GPT-2 è pubblico)")

verifica_chiave("HUGGINGFACE_TOKEN", HF_TOKEN)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"\nDevice in uso: {DEVICE}")
if DEVICE == "cpu":
    print("Nessuna GPU rilevata: va bene, il modello e il dataset di questo notebook sono piccoli.")


---
## Cella 1 — Carichiamo GPT-2 e contiamo i parametri

Prima di toccare LoRA, stabiliamo la baseline: quanti parametri ha GPT-2 in totale, e quanti ne
aggiorneremmo con un full fine-tuning (Modulo 17)? Scriviamo una funzione di conteggio che
riuseremo più avanti per confrontare la situazione prima/dopo LoRA (è l'equivalente in italiano
della funzione `print_trainable_parameters` che trovi nel materiale sorgente di questo modulo).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

NOME_MODELLO = "gpt2"  # 124M parametri, pubblico

tokenizer = AutoTokenizer.from_pretrained(NOME_MODELLO)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 non ha un token di padding dedicato

modello = AutoModelForCausalLM.from_pretrained(NOME_MODELLO)


def conta_parametri(modello) -> dict:
    """Conta i parametri totali e quelli addestrabili (requires_grad=True) di un modello.
    Su un modello appena caricato, senza nulla congelato, i due numeri coincidono."""
    totali = 0
    addestrabili = 0
    for parametro in modello.parameters():
        n = parametro.numel()
        totali += n
        if parametro.requires_grad:
            addestrabili += n
    percentuale = 100 * addestrabili / totali
    return {
        "parametri_totali": totali,
        "parametri_addestrabili": addestrabili,
        "percentuale_addestrabile": round(percentuale, 4),
    }


info_base = conta_parametri(modello)
print("=== GPT-2 prima di LoRA (modello \"nudo\") ===")
for chiave, valore in info_base.items():
    print(f"  {chiave}: {valore:,}" if isinstance(valore, int) else f"  {chiave}: {valore}%")

print(
    "\nQuesto e' il numero di parametri che aggiorneresti con un full fine-tuning (Modulo 17): "
    "tutti quanti, il 100%."
)


---
## Cella 2 — Il modello base prima del fine-tuning

Proviamo a chiedere a GPT-2 "nudo" di fare esattamente il compito della dispensa (Sezione 5):
dato un articolo, generare titolo, parole chiave SEO e riassunto in un JSON con un formato
preciso.

**Prima di eseguire:** GPT-2 non ha mai visto istruzioni di questo tipo durante il
preaddestramento (a differenza di un modello instruction-tuned). Cosa ti aspetti che generi?

In [ ]:
def genera_testo(prompt: str, modello, tokenizer, max_nuovi_token: int = 60) -> str:
    """Genera testo a partire da un prompt, usando greedy decoding (deterministico)
    per rendere i confronti prima/dopo più facili da leggere."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(modello.device)
    output = modello.generate(
        input_ids,
        max_new_tokens=max_nuovi_token,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)


PROMPT_ESEMPIO = (
    "Articolo: Il comune ha annunciato l'apertura di tre nuove piste ciclabili nel centro "
    "storico, con l'obiettivo di ridurre il traffico automobilistico del 15% entro il 2027.\n"
    "Genera titolo, parole chiave SEO e riassunto in formato JSON.\n"
    "Output:"
)

print(genera_testo(PROMPT_ESEMPIO, modello, tokenizer))


**Cosa osservare:** con tutta probabilità GPT-2 continua il testo in modo plausibile a livello
grammaticale, ma non produce nulla che somigli a un JSON con le tre chiavi richieste — perché non
ha mai imparato ad associare quella istruzione a quel formato di output. È esattamente il
problema che la dispensa descrive nella Sezione 5: il fine-tuning serve a imporre il formato e lo
stile, non solo a "sapere cose".

---
## Cella 3 — Il mini-dataset in italiano

Costruiamo un dataset dimostrativo con lo stesso spirito della Sezione 5.1 della dispensa: un
insieme (piccolo, qui a scopo didattico) di coppie "articolo → output JSON atteso", per una
testata immaginaria che vuole un formato fisso: `titolo`, `seo` (lista di parole chiave) e
`riassunto`.

> **Nota:** con una decina di esempi verifichiamo solo che il meccanismo funzioni. Per un
> adattamento stabile su un caso reale la dispensa indica 500-1.000 esempi (Sezione 5.1).

In [ ]:
ESEMPI = [
    {
        "articolo": "Il comune ha annunciato tre nuove piste ciclabili nel centro storico, con l'obiettivo di ridurre il traffico del 15% entro il 2027.",
        "output": '{"titolo": "Tre nuove piste ciclabili nel centro storico", "seo": ["piste ciclabili", "mobilita urbana", "traffico"], "riassunto": "Il comune apre tre piste ciclabili per ridurre il traffico del 15% entro il 2027."}',
    },
    {
        "articolo": "Una startup locale ha raccolto due milioni di euro per sviluppare sensori agricoli a basso costo destinati alle piccole aziende.",
        "output": '{"titolo": "Due milioni di euro per i sensori agricoli", "seo": ["startup", "agricoltura", "sensori"], "riassunto": "Una startup raccoglie due milioni di euro per sensori agricoli economici."}',
    },
    {
        "articolo": "L'ospedale cittadino ha inaugurato un nuovo reparto di radiologia con due macchinari di ultima generazione.",
        "output": '{"titolo": "Nuovo reparto di radiologia in ospedale", "seo": ["ospedale", "radiologia", "sanita"], "riassunto": "Inaugurato il nuovo reparto di radiologia con due macchinari di ultima generazione."}',
    },
    {
        "articolo": "La squadra locale ha vinto la finale del campionato regionale davanti a duemila spettatori.",
        "output": '{"titolo": "Vittoria nella finale del campionato regionale", "seo": ["sport", "campionato", "finale"], "riassunto": "La squadra locale vince la finale davanti a duemila spettatori."}',
    },
    {
        "articolo": "Il festival del cinema indipendente tornera' a settembre con oltre cinquanta proiezioni in tre sale.",
        "output": '{"titolo": "Torna il festival del cinema indipendente", "seo": ["cinema", "festival", "cultura"], "riassunto": "A settembre il festival del cinema indipendente propone oltre cinquanta proiezioni."}',
    },
    {
        "articolo": "Un'azienda tessile della zona ha convertito una linea di produzione per usare solo cotone riciclato.",
        "output": '{"titolo": "Cotone riciclato nella produzione tessile locale", "seo": ["tessile", "sostenibilita", "riciclo"], "riassunto": "Un'azienda locale converte una linea di produzione al cotone riciclato."}',
    },
    {
        "articolo": "La biblioteca comunale ha esteso l'orario serale di apertura fino alle 22 per tre giorni a settimana.",
        "output": '{"titolo": "Orario serale esteso alla biblioteca comunale", "seo": ["biblioteca", "orari", "servizi comunali"], "riassunto": "La biblioteca comunale apre fino alle 22 tre giorni a settimana."}',
    },
    {
        "articolo": "Un gruppo di ricercatori universitari ha sviluppato un algoritmo per prevedere le ondate di calore con tre giorni di anticipo.",
        "output": '{"titolo": "Un algoritmo per prevedere le ondate di calore", "seo": ["ricerca", "clima", "universita"], "riassunto": "Ricercatori universitari sviluppano un algoritmo che prevede le ondate di calore con tre giorni di anticipo."}',
    },
    {
        "articolo": "Il mercato coperto riaprira' dopo sei mesi di lavori di ristrutturazione con venti nuovi banchi.",
        "output": '{"titolo": "Riapre il mercato coperto dopo la ristrutturazione", "seo": ["mercato", "commercio locale", "ristrutturazione"], "riassunto": "Dopo sei mesi di lavori il mercato coperto riapre con venti nuovi banchi."}',
    },
    {
        "articolo": "L'aeroporto cittadino ha attivato due nuove rotte dirette verso capitali europee a partire da giugno.",
        "output": '{"titolo": "Due nuove rotte dirette dall\'aeroporto cittadino", "seo": ["aeroporto", "voli", "turismo"], "riassunto": "Da giugno attive due nuove rotte dirette verso capitali europee."}',
    },
]

print(f"Numero di esempi nel dataset: {len(ESEMPI)}")
print("\nPrimo esempio:")
print(ESEMPI[0]["articolo"])
print("->", ESEMPI[0]["output"])


In [ ]:
from datasets import Dataset

ISTRUZIONE = "Genera titolo, parole chiave SEO e riassunto in formato JSON."


def costruisci_testo_addestramento(esempio: dict) -> str:
    """Compone il testo completo che GPT-2 vedrà in addestramento: prompt + output atteso,
    terminato dal token di fine sequenza (così il modello impara anche QUANDO fermarsi)."""
    return (
        f"Articolo: {esempio['articolo']}\n"
        f"{ISTRUZIONE}\n"
        f"Output: {esempio['output']}" + tokenizer.eos_token
    )


testi_addestramento = [costruisci_testo_addestramento(e) for e in ESEMPI]
dataset = Dataset.from_dict({"text": testi_addestramento})


def tokenizza(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256, padding="max_length")


dataset_tokenizzato = dataset.map(tokenizza, batched=True, remove_columns=["text"])
print(dataset_tokenizzato)


---
## Cella 4 — Configuriamo LoRA con `peft`

Applichiamo LoRA alla matrice di attenzione di GPT-2. In GPT-2 le proiezioni *query*, *key* e
*value* (Modulo 15) sono combinate in un'unica matrice chiamata `c_attn`: è il target che
indichiamo in `target_modules`. I parametri `r` e `lora_alpha` sono quelli descritti nella
dispensa (Sezione 3.3 e 3.6): partiamo da valori bassi, `r=8`, come consigliato.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

configurazione_lora = LoraConfig(
    r=8,                          # rank: dimensione del "collo di bottiglia" tra le matrici A e B
    lora_alpha=16,                # fattore di scala (alpha/r nella formula della dispensa, Sezione 3.3)
    target_modules=["c_attn"],    # in GPT-2, Q/K/V sono combinate in questa unica matrice
    lora_dropout=0.05,            # dropout di regolarizzazione solo sui parametri LoRA
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

modello_lora = get_peft_model(modello, configurazione_lora)

# peft mette a disposizione un metodo pronto per questo identico conteggio
modello_lora.print_trainable_parameters()

info_lora = conta_parametri(modello_lora)
print("\n=== GPT-2 dopo LoRA (stessa funzione di conteggio della Cella 1) ===")
for chiave, valore in info_lora.items():
    print(f"  {chiave}: {valore:,}" if isinstance(valore, int) else f"  {chiave}: {valore}%")


**Cosa osservare:** la percentuale di parametri addestrabili che vedrai è più alta del ~0,2%
citato nella dispensa (Sezione 3.4) per un modello LLaMA da diversi miliardi di parametri. Non è
un errore: è la "regola pratica" della Sezione 3.4 applicata in senso inverso — GPT-2 è molto più
piccolo, quindi la stessa matrice `c_attn` pesa proporzionalmente di più sul totale del modello.
Più il modello di partenza è grande, più il risparmio relativo di LoRA cresce.

---
## Cella 5 — Esperimento guidato: verifica numerica sulla matrice reale

Rifacciamo, con codice, lo stesso calcolo dell'esempio numerico della dispensa (Sezione 3.4:
matrice 512×64, r=8 → ~14%), mostrando che il funziona identico applicato alla forma **reale**
della matrice `c_attn` di GPT-2 (768 in ingresso, 3×768=2304 in uscita perché contiene Q, K e V
concatenate).

In [ ]:
def confronta_full_finetuning_vs_lora(d: int, k: int, r: int) -> dict:
    """Dati le dimensioni d x k di una matrice di peso e un rank r, calcola quanti parametri
    aggiornerebbe un full fine-tuning su quella matrice, quanti ne introduce LoRA (matrici A e B),
    e la percentuale relativa — stesso calcolo della Sezione 3.4 della dispensa."""
    parametri_full = d * k
    parametri_a = d * r
    parametri_b = r * k
    parametri_lora = parametri_a + parametri_b
    percentuale = 100 * parametri_lora / parametri_full
    return {
        "dimensione_matrice": f"{d}x{k}",
        "parametri_full_finetuning": parametri_full,
        "parametri_lora_A": parametri_a,
        "parametri_lora_B": parametri_b,
        "parametri_lora_totali": parametri_lora,
        "percentuale_rispetto_a_full": round(percentuale, 2),
    }


print("=== Esempio della dispensa (matrice 512x64, r=8) ===")
for k, v in confronta_full_finetuning_vs_lora(d=512, k=64, r=8).items():
    print(f"  {k}: {v}")

print("\n=== La matrice reale c_attn di GPT-2 (768x2304, r=8) ===")
for k, v in confronta_full_finetuning_vs_lora(d=768, k=2304, r=8).items():
    print(f"  {k}: {v}")

print(
    "\nNota: il risparmio percentuale sulla singola matrice qui e' minore che nell'esempio della "
    "dispensa, perche' la matrice di GPT-2 e' 'piu' larga in una dimensione (2304 invece di 64) "
    "-- coerente con la regola pratica della Sezione 3.4: la convenienza di LoRA dipende dalla "
    "forma delle matrici del modello, non solo dalla sua dimensione totale."
)


---
## Cella 6 — Alleniamo l'adattatore LoRA

Un ciclo di addestramento breve, con `Trainer` di `transformers`. Con solo 10 esempi e pochi
epoch il modello non diventerà affidabile in produzione (la dispensa è chiara sul punto, Sezione
5.1), ma è sufficiente per osservare che il meccanismo funziona: la loss deve scendere.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

argomenti_addestramento = TrainingArguments(
    output_dir="./gpt2-lora-editoriale-checkpoint",
    per_device_train_batch_size=2,
    num_train_epochs=15,       # dataset minuscolo: piu' epoch del solito per farlo convergere
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    report_to=[],
)

trainer = Trainer(
    model=modello_lora,
    args=argomenti_addestramento,
    train_dataset=dataset_tokenizzato,
    data_collator=collator,
)

risultato_training = trainer.train()
print("\nLoss finale:", risultato_training.training_loss)


---
## Cella 7 — Confronto: prima vs dopo il fine-tuning LoRA

Rilanciamo lo stesso prompt della Cella 2 e confrontiamo direttamente.

In [ ]:
print("=== PRIMA di LoRA (rigenera il modello base per confronto pulito) ===")
modello_base_fresco = AutoModelForCausalLM.from_pretrained(NOME_MODELLO)
print(genera_testo(PROMPT_ESEMPIO, modello_base_fresco, tokenizer))

print("\n=== DOPO LoRA ===")
print(genera_testo(PROMPT_ESEMPIO, modello_lora, tokenizer))


**Le tue osservazioni** (modifica questa cella):

- Prima del fine-tuning, il modello generava: ...
- Dopo il fine-tuning, il formato si avvicina di più al JSON atteso? In che modo? ...
- Cosa cambierebbe, secondo te, aggiungendo più esempi al dataset (vedi Sezione 5.1 della
  dispensa: nella pratica servono 500-1.000 esempi)? ...

---
## Esercizio — completa il codice

**TODO:** aggiungi almeno 3 nuovi esempi (articolo + output JSON) alla lista `ESEMPI` della
Cella 3, ricostruisci il dataset e rilancia l'addestramento (Cella 6) e il confronto (Cella 7).
Il modello migliora, peggiora, o resta uguale con più dati? Scrivi la tua risposta nella cella
markdown qui sotto.

In [ ]:
# TODO: aggiungi qui almeno 3 nuovi esempi nello stesso formato di ESEMPI, poi:
# 1) ricrea `testi_addestramento`, `dataset`, `dataset_tokenizzato` (Cella 3)
# 2) rilancia il Trainer (Cella 6) -- puoi ripartire da un modello_lora fresco per un confronto pulito
# 3) confronta di nuovo l'output (Cella 7)

nuovi_esempi = [
    # {"articolo": "...", "output": '{"titolo": "...", "seo": [...], "riassunto": "..."}'},
]


**Le tue osservazioni sull'esercizio** (modifica questa cella):

- Con più esempi, il formato JSON è: ...
- Cosa ho notato di diverso: ...

---
## Sfida finale — Salva, confronta le dimensioni, ricarica

La Sezione 5.3 della dispensa spiega che a fine addestramento non si salva il modello completo,
ma solo le matrici LoRA — un file di poche centinaia di kilobyte contro le centinaia di megabyte
del modello base. Verifichiamolo con codice, e costruiamo una funzione che ricarica
base+adattatore per l'inferenza (lo scenario "adattatore separato" descritto nella Sezione 3.5).

Completa i `# TODO`.

In [ ]:
import os

CARTELLA_ADATTATORE = "./adattatore-gpt2-editoriale"
CARTELLA_MODELLO_BASE = "./gpt2-base-salvato"


def dimensione_cartella_mb(percorso: str) -> float:
    """Somma la dimensione di tutti i file in una cartella, in megabyte."""
    totale_byte = 0
    for cartella_corrente, _, file in os.walk(percorso):
        for f in file:
            totale_byte += os.path.getsize(os.path.join(cartella_corrente, f))
    return totale_byte / (1024 * 1024)


def salva_e_confronta_dimensioni(modello_lora, modello_base, cartella_adattatore: str, cartella_base: str) -> None:
    """Salva SOLO l'adattatore LoRA e, separatamente, il modello base intero,
    poi confronta le dimensioni su disco -- la stessa verifica descritta nella
    Sezione 5.3 della dispensa."""
    # TODO: salva l'adattatore con modello_lora.save_pretrained(cartella_adattatore)
    # TODO: salva il modello base intero con modello_base.save_pretrained(cartella_base)

    dimensione_adattatore = dimensione_cartella_mb(cartella_adattatore)
    dimensione_base = dimensione_cartella_mb(cartella_base)

    print(f"Dimensione adattatore LoRA: {dimensione_adattatore:.2f} MB")
    print(f"Dimensione modello base completo: {dimensione_base:.2f} MB")
    if dimensione_adattatore > 0:
        print(f"Rapporto: il modello base e' circa {dimensione_base / dimensione_adattatore:.0f}x piu' grande")


def carica_base_piu_adattatore(nome_modello_base: str, cartella_adattatore: str):
    """Ricarica un modello base 'pulito' e vi inietta l'adattatore salvato --
    lo scenario a cui la dispensa fa riferimento in Sezione 3.5: un solo modello
    base in memoria, adattatori intercambiabili al volo."""
    from peft import PeftModel

    base_fresca = AutoModelForCausalLM.from_pretrained(nome_modello_base)
    # TODO: usa PeftModel.from_pretrained(base_fresca, cartella_adattatore) per iniettare l'adattatore
    modello_con_adattatore = None
    return modello_con_adattatore


# Decommenta dopo aver completato i TODO sopra
# salva_e_confronta_dimensioni(modello_lora, modello_base_fresco, CARTELLA_ADATTATORE, CARTELLA_MODELLO_BASE)
# modello_ricaricato = carica_base_piu_adattatore(NOME_MODELLO, CARTELLA_ADATTATORE)
# print(genera_testo(PROMPT_ESEMPIO, modello_ricaricato, tokenizer))


### Estensione opzionale (se hai tempo)

Prova `modello_con_adattatore.merge_and_unload()` (metodo di `peft`) e richiama
`conta_parametri` sul risultato: dovresti ritrovare esattamente lo stesso numero di parametri
totali del modello base originale — è la fusione descritta nella Sezione 3.5 e 5.3 della
dispensa, quella che non aggiunge alcun costo in inferenza.

---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 21 — Modulo 21 del Corso Completo di IA Generativa*